# Week 7 Community Contribution - BernardUdo

This notebook tracks the Week 7 end-to-end open-source fine-tuning workflow.

## Scope by day
- Day 1: Define project task and model strategy (QLoRA/LoRA).
- Day 2: Build prompt-completion data and split datasets.
- Day 3-4: Fine-tune an open-source model.
- Day 5: Evaluate on held-out data and summarize performance.

In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd()
REPO_ROOT = next((p for p in [PROJECT_DIR, *PROJECT_DIR.parents] if (p / ".git").exists()), PROJECT_DIR)
WEEK7_DIR = REPO_ROOT / "week7"

if str(WEEK7_DIR) not in sys.path:
    sys.path.append(str(WEEK7_DIR))

print(f"Project dir: {PROJECT_DIR}")
print(f"Repo root:   {REPO_ROOT}")
print(f"Week7 dir:   {WEEK7_DIR}")

## Run order
1. Install dependencies in your notebook environment.
2. Load curated Week 7 item data from Hugging Face Hub.
3. Build prompt-completion pairs for SFT.
4. Fine-tune an open-source model (QLoRA style config).
5. Evaluate on held-out test prompts and summarize results.

## Day 1-2: Environment and data setup

If needed, install dependencies in this kernel:

```bash
pip install -U datasets transformers peft trl accelerate bitsandbytes scikit-learn pandas matplotlib
```

> Tip: for GPU training, run this notebook in Colab with a T4/L4/A100 runtime.

In [ ]:
import re
from statistics import mean

import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer

from pricer.items import Item

In [ ]:
# Change USERNAME when you are ready to push processed prompt data to your own Hugging Face account.
USERNAME = "ed-donner"
LITE_MODE = True
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
MAX_SUMMARY_TOKENS = 220
DO_ROUND = True
SEED = 42

raw_dataset_name = f"{USERNAME}/items_lite" if LITE_MODE else f"{USERNAME}/items_full"
prompt_dataset_name = f"{USERNAME}/items_prompts_lite" if LITE_MODE else f"{USERNAME}/items_prompts_full"

print("raw_dataset_name:", raw_dataset_name)
print("prompt_dataset_name:", prompt_dataset_name)
print("base_model:", BASE_MODEL)

In [ ]:
train_items, val_items, test_items = Item.from_hub(raw_dataset_name)
print(f"train={len(train_items)}, val={len(val_items)}, test={len(test_items)}")
print(train_items[0])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

for item in train_items:
    item.make_prompts(tokenizer=tokenizer, max_tokens=MAX_SUMMARY_TOKENS, do_round=DO_ROUND)
for item in val_items:
    item.make_prompts(tokenizer=tokenizer, max_tokens=MAX_SUMMARY_TOKENS, do_round=DO_ROUND)
for item in test_items:
    item.make_prompts(tokenizer=tokenizer, max_tokens=MAX_SUMMARY_TOKENS, do_round=DO_ROUND)

print(train_items[0].prompt[:300])
print("completion:", train_items[0].completion)

In [ ]:
# Optional: publish prompt-completion data to your own HF dataset namespace.
# Set USERNAME to your HF user/org before running this.
# Item.push_prompts_to_hub(prompt_dataset_name, train_items, val_items, test_items)

## Day 3-4: Fine-tuning workflow (QLoRA-style)

This section is intentionally compact and reproducible. You can run it on GPU in Colab.

If your environment cannot install `bitsandbytes`, keep the code and run it in Colab for the actual training step.

In [ ]:
def to_sft_dataset(items):
    rows = []
    for item in items:
        text = f"{item.prompt}{item.completion}"
        rows.append({"text": text})
    return Dataset.from_list(rows)

train_sft = to_sft_dataset(train_items)
val_sft = to_sft_dataset(val_items)

train_sft[0]

In [ ]:
# Fine-tuning cell (run on GPU runtime)
try:
    import torch
    from peft import LoraConfig
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
    from trl import SFTTrainer

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        task_type="CAUSAL_LM",
    )

    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
    )

    training_args = TrainingArguments(
        output_dir="./artifacts/week7_qlora",
        learning_rate=2e-4,
        num_train_epochs=1,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        logging_steps=10,
        save_steps=100,
        eval_strategy="steps",
        eval_steps=100,
        report_to="none",
    )

    trainer = SFTTrainer(
        model=model,
        train_dataset=train_sft,
        eval_dataset=val_sft,
        peft_config=lora_config,
        dataset_text_field="text",
        args=training_args,
    )

    # Uncomment to train on GPU:
    # trainer.train()
    # trainer.save_model("./artifacts/week7_qlora/final")
    print("QLoRA scaffold initialized. Uncomment training lines to run on GPU runtime.")
except Exception as exc:
    print(f"Skipping fine-tuning scaffold setup in this environment: {exc}")
    print("Run this cell in Colab/local CUDA with torch + bitsandbytes + peft + trl installed.")

## Day 5: Evaluate on held-out test set

This section gives you a guaranteed runnable baseline, and an optional fine-tuned model evaluator if you trained an adapter/model.

In [ ]:
def extract_price(text_or_value):
    if isinstance(text_or_value, (int, float)):
        return float(text_or_value)
    text = str(text_or_value).replace(",", "")
    match = re.search(r"[-+]?\d*\.\d+|\d+", text)
    return float(match.group()) if match else 0.0

mean_train_price = mean([item.price for item in train_items])

def baseline_predictor(_item):
    return mean_train_price

print(f"Baseline mean price: ${mean_train_price:,.2f}")

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def score_predictor(predict_fn, items, size=200):
    subset = items[: min(size, len(items))]
    y_true = np.array([item.price for item in subset], dtype=float)
    y_pred = np.array([extract_price(predict_fn(item)) for item in subset], dtype=float)
    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(mean_squared_error(y_true, y_pred) ** 0.5)
    r2 = float(r2_score(y_true, y_pred))
    within_20 = float((np.abs(y_pred - y_true) <= 20).mean() * 100)
    return {
        "mae": mae,
        "rmse": rmse,
        "r2": r2,
        "within_20_pct": within_20,
    }

baseline_metrics = score_predictor(baseline_predictor, test_items, size=200)
baseline_metrics

In [ ]:
# Optional fine-tuned evaluation
# Set this path to your trained adapter/model folder, then run this cell.
# Example: FINETUNED_MODEL_PATH = "./artifacts/week7_qlora/final"
FINETUNED_MODEL_PATH = None
finetuned_metrics = None

if FINETUNED_MODEL_PATH:
    from transformers import AutoModelForCausalLM, pipeline

    ft_model = AutoModelForCausalLM.from_pretrained(FINETUNED_MODEL_PATH, device_map="auto")
    ft_pipe = pipeline("text-generation", model=ft_model, tokenizer=tokenizer)

    def finetuned_predictor(item):
        prompt = item.prompt
        out = ft_pipe(prompt, max_new_tokens=8, do_sample=False)[0]["generated_text"]
        return out[len(prompt):]

    finetuned_metrics = score_predictor(finetuned_predictor, test_items, size=200)
    print(finetuned_metrics)
else:
    print("Set FINETUNED_MODEL_PATH to run fine-tuned model evaluation.")

In [ ]:
import pandas as pd

rows = [{"model": "baseline", **baseline_metrics}]
if finetuned_metrics is not None:
    rows.append({"model": "finetuned", **finetuned_metrics})

comparison_df = pd.DataFrame(rows)
comparison_df

## Submission summary

- **Base model:** `Qwen/Qwen2.5-0.5B-Instruct`
- **Fine-tuning method:** LoRA/QLoRA-style SFT scaffold in notebook
- **Dataset used:** `ed-donner/items_lite` with prompt-completion conversion
- **Current lite-run baseline metrics (test subset size=200):**
  - MAE: `106.08`
  - RMSE: `148.35`
  - R2: `-0.0014`
  - % within $20: `10.0`
- **Notes:** baseline and full pipeline scaffolding are executed; fine-tuning run is intentionally deferred to GPU environment (Colab/local CUDA) by enabling the training lines in the Day 3-4 cell.